# Emissions Dashboard — Full Pipeline
**Datasets:** FAO EMS WIDEF (agrifood emissions share) + JRC EDGAR WIDEF (air pollutant tonnes)

**Steps covered:**
1. Install & import
2. Load datasets
3. Explore & understand
4. Clean FAO dataset
5. Clean JRC dataset
6. Join datasets
7. Export cleaned files
8. Dashboard (Plotly)


## Step 1 — Install & Import

In [1]:
# Run this cell first — installs libraries not pre-installed in Colab
!pip install plotly --quiet
!pip install kaleido --quiet   # needed to export Plotly charts as images

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 69.0/69.0 kB 5.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 49.3/49.3 kB 3.2 MB/s eta 0:00:00


In [2]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [3]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import plotly.express as px
import plotly.graph_objects as go
from plotly.subplots import make_subplots

pd.set_option('display.max_columns', 30)
pd.set_option('display.float_format', '{:.4f}'.format)
print('All libraries loaded successfully.')

All libraries loaded successfully.


## Step 2 — Load the Datasets

> Upload both CSV files using the Colab file panel (folder icon on the left sidebar), then run this cell.

In [4]:
# If running in Google Colab, upload files first:
# from google.colab import files
# files.upload()   # uncomment and run to upload interactively

fao_raw = pd.read_csv('/content/drive/MyDrive/ICW/FAO_EMS_WIDEF (1).csv')
jrc_raw = pd.read_csv('/content/drive/MyDrive/ICW/JRC_EDGAR_WIDEF.csv')

print('FAO shape:', fao_raw.shape)   # expect (2859, 52)
print('JRC shape:', jrc_raw.shape)   # expect (2844, 77)

FAO shape: (2859, 52)
JRC shape: (2844, 77)


## Step 3 — Explore & Understand

In [5]:
# ── FAO: what does it contain? ──────────────────────────────────
print('=== FAO columns ===')
print(fao_raw.columns.tolist())

print('\n=== FAO: unique indicators ===')
print(fao_raw[['INDICATOR','INDICATOR_LABEL']].drop_duplicates())

print('\n=== FAO: sector breakdown (COMP_BREAKDOWN_1) ===')
print(fao_raw[['COMP_BREAKDOWN_1','COMP_BREAKDOWN_1_LABEL']].drop_duplicates().to_string())

print('\n=== FAO: unit of measurement ===')
print(fao_raw[['UNIT_MEASURE','UNIT_MEASURE_LABEL']].drop_duplicates())
# PT = Percentage — all values are % shares of total emissions

=== FAO columns ===
['FREQ', 'FREQ_LABEL', 'REF_AREA', 'REF_AREA_LABEL', 'INDICATOR', 'INDICATOR_LABEL', 'UNIT_MEASURE', 'UNIT_MEASURE_LABEL', 'COMP_BREAKDOWN_1', 'COMP_BREAKDOWN_1_LABEL', 'DATABASE_ID', 'DATABASE_ID_LABEL', 'UNIT_MULT', 'UNIT_MULT_LABEL', 'OBS_STATUS', 'OBS_STATUS_LABEL', 'OBS_CONF', 'OBS_CONF_LABEL', '1990', '1991', '1992', '1993', '1994', '1995', '1996', '1997', '1998', '1999', '2000', '2001', '2002', '2003', '2004', '2005', '2006', '2007', '2008', '2009', '2010', '2011', '2012', '2013', '2014', '2015', '2016', '2017', '2018', '2019', '2020', '2021', '2022', '2023']

=== FAO: unique indicators ===
        INDICATOR                INDICATOR_LABEL
0  FAO_EMS_726313  Emissions Share (CO2eq) (AR5)

=== FAO: sector breakdown (COMP_BREAKDOWN_1) ===
   COMP_BREAKDOWN_1                COMP_BREAKDOWN_1_LABEL
0     FAO_ITEM_1707                          Item: LULUCF
1     FAO_ITEM_1711                Item: IPCC Agriculture
2     FAO_ITEM_5084            Item: Emissions from c

In [6]:
# ── JRC: what does it contain? ──────────────────────────────────
print('=== JRC columns ===')
print(jrc_raw.columns.tolist())

print('\n=== JRC: unique indicators (pollutants) ===')
print(jrc_raw[['INDICATOR','INDICATOR_LABEL']].drop_duplicates().to_string())

print('\n=== JRC: breakdown categories ===')
print(jrc_raw[['COMP_BREAKDOWN_1','COMP_BREAKDOWN_1_LABEL']].drop_duplicates())
# _T = Total (most rows). Mercury has 3 sub-species (HG_D, HG_G, HG_P)

print('\n=== JRC: unit of measurement ===')
print(jrc_raw[['UNIT_MEASURE','UNIT_MEASURE_LABEL']].drop_duplicates())
# T = Tonnes

=== JRC columns ===
['FREQ', 'FREQ_LABEL', 'REF_AREA', 'REF_AREA_LABEL', 'INDICATOR', 'INDICATOR_LABEL', 'UNIT_MEASURE', 'UNIT_MEASURE_LABEL', 'COMP_BREAKDOWN_1', 'COMP_BREAKDOWN_1_LABEL', 'AGG_METHOD', 'AGG_METHOD_LABEL', 'DECIMALS', 'DECIMALS_LABEL', 'DATABASE_ID', 'DATABASE_ID_LABEL', 'UNIT_MULT', 'UNIT_MULT_LABEL', 'DATA_SOURCE', 'DATA_SOURCE_LABEL', 'OBS_STATUS', 'OBS_STATUS_LABEL', 'OBS_CONF', 'OBS_CONF_LABEL', '1970', '1971', '1972', '1973', '1974', '1975', '1976', '1977', '1978', '1979', '1980', '1981', '1982', '1983', '1984', '1985', '1986', '1987', '1988', '1989', '1990', '1991', '1992', '1993', '1994', '1995', '1996', '1997', '1998', '1999', '2000', '2001', '2002', '2003', '2004', '2005', '2006', '2007', '2008', '2009', '2010', '2011', '2012', '2013', '2014', '2015', '2016', '2017', '2018', '2019', '2020', '2021', '2022']

=== JRC: unique indicators (pollutants) ===
          INDICATOR                                   INDICATOR_LABEL
0      JRC_EDGAR_BC            Carbonace

In [7]:
# ── Missing value overview ───────────────────────────────────────
year_cols_fao = [c for c in fao_raw.columns if c.isdigit() or (len(c)==4 and c.startswith(('1','2')))]
year_cols_jrc = [c for c in jrc_raw.columns if c.isdigit() or (len(c)==4 and c.startswith(('1','2')))]

fao_null_pct = fao_raw[year_cols_fao].isna().mean().mean() * 100
jrc_null_pct = jrc_raw[year_cols_jrc].isna().mean().mean() * 100

print(f'FAO missing values in year columns: {fao_null_pct:.1f}%')
print(f'JRC missing values in year columns: {jrc_null_pct:.1f}%')

# ── Country overlap ──────────────────────────────────────────────
fao_countries = set(fao_raw['REF_AREA'].unique())
jrc_countries = set(jrc_raw['REF_AREA'].unique())
common = fao_countries & jrc_countries

print(f'\nFAO countries:  {len(fao_countries)}')
print(f'JRC countries:  {len(jrc_countries)}')
print(f'Common:         {len(common)}')
print(f'Only in FAO:    {len(fao_countries - jrc_countries)}')
print(f'Only in JRC:    {len(jrc_countries - fao_countries)}')

FAO missing values in year columns: 2.7%
JRC missing values in year columns: 0.5%

FAO countries:  206
JRC countries:  222
Common:         200
Only in FAO:    6
Only in JRC:    22


## Step 4 — Clean the FAO Dataset

In [8]:
# Work on a copy — never mutate the raw data
fao = fao_raw.copy()

# ── 4a. Drop metadata columns that add no analytical value ───────
# These are either constant, redundant labels, or observation flags
fao_drop = [
    'FREQ', 'FREQ_LABEL',          # always 'A' / 'Annual'
    'REF_AREA_LABEL',              # country name — we have REF_AREA code
    'INDICATOR_LABEL',             # always same string
    'UNIT_MEASURE_LABEL',          # always 'Percentage'
    'COMP_BREAKDOWN_1_LABEL',      # human label for the code we keep
    'DATABASE_ID', 'DATABASE_ID_LABEL',
    'UNIT_MULT', 'UNIT_MULT_LABEL',
    'OBS_STATUS', 'OBS_STATUS_LABEL',
    'OBS_CONF', 'OBS_CONF_LABEL'
]
fao = fao.drop(columns=fao_drop)

print('FAO columns after drop:', fao.columns.tolist())
print('Shape:', fao.shape)

FAO columns after drop: ['REF_AREA', 'INDICATOR', 'UNIT_MEASURE', 'COMP_BREAKDOWN_1', '1990', '1991', '1992', '1993', '1994', '1995', '1996', '1997', '1998', '1999', '2000', '2001', '2002', '2003', '2004', '2005', '2006', '2007', '2008', '2009', '2010', '2011', '2012', '2013', '2014', '2015', '2016', '2017', '2018', '2019', '2020', '2021', '2022', '2023']
Shape: (2859, 38)


In [9]:
# ── 4b. Melt from wide → long (tidy format) ─────────────────────
# Currently: one row per country-sector, years as columns
# Target:    one row per country-sector-year

id_vars_fao = ['REF_AREA', 'INDICATOR', 'UNIT_MEASURE', 'COMP_BREAKDOWN_1']
year_cols   = [c for c in fao.columns if c not in id_vars_fao]

fao_long = fao.melt(
    id_vars   = id_vars_fao,
    value_vars = year_cols,
    var_name   = 'year',
    value_name = 'value'
)

# ── 4c. Fix data types ───────────────────────────────────────────
fao_long['year']  = fao_long['year'].astype(int)
fao_long['value'] = pd.to_numeric(fao_long['value'], errors='coerce')

# ── 4d. Drop nulls ───────────────────────────────────────────────
before = len(fao_long)
fao_long = fao_long.dropna(subset=['value'])
after  = len(fao_long)
print(f'FAO rows dropped (null value): {before - after} ({(before-after)/before*100:.1f}%)')
print(f'FAO long shape: {fao_long.shape}')
fao_long.head()

FAO rows dropped (null value): 2606 (2.7%)
FAO long shape: (94600, 6)


,REF_AREA,INDICATOR,UNIT_MEASURE,COMP_BREAKDOWN_1,year,value
0,ABW,FAO_EMS_726313,PT,FAO_ITEM_1707,1990,0.0600
1,ABW,FAO_EMS_726313,PT,FAO_ITEM_1711,1990,0.0000
3,ABW,FAO_EMS_726313,PT,FAO_ITEM_6517,1990,21.0300
4,ABW,FAO_EMS_726313,PT,FAO_ITEM_6518,1990,21.0300
5,ABW,FAO_EMS_726313,PT,FAO_ITEM_6818,1990,2.5400


In [10]:
# ── 4e. Add a readable sector label column ───────────────────────
sector_labels = {
    'FAO_ITEM_1707': 'LULUCF',
    'FAO_ITEM_1711': 'IPCC Agriculture',
    'FAO_ITEM_5084': 'Emissions from crops',
    'FAO_ITEM_5085': 'Emissions from livestock',
    'FAO_ITEM_6517': 'Pre- and Post-Production',
    'FAO_ITEM_6518': 'Agrifood systems',
    'FAO_ITEM_6818': 'Waste',
    'FAO_ITEM_6819': 'Other',
    'FAO_ITEM_6821': 'Energy',
    'FAO_ITEM_6824': 'AFOLU',
    'FAO_ITEM_6825': 'All sectors with LULUCF',
    'FAO_ITEM_6829': 'All sectors without LULUCF',
    'FAO_ITEM_6995': 'Emissions on agricultural land',
    'FAO_ITEM_6996': 'Farm gate'
}
fao_long['sector'] = fao_long['COMP_BREAKDOWN_1'].map(sector_labels)

# ── 4f. Add source tag ───────────────────────────────────────────
fao_long['source'] = 'FAO'

print('FAO cleaned. Sample:')
fao_long.head(3)

FAO cleaned. Sample:


,REF_AREA,INDICATOR,UNIT_MEASURE,COMP_BREAKDOWN_1,year,value,sector,source
0,ABW,FAO_EMS_726313,PT,FAO_ITEM_1707,1990,0.0600,LULUCF,FAO
1,ABW,FAO_EMS_726313,PT,FAO_ITEM_1711,1990,0.0000,IPCC Agriculture,FAO
3,ABW,FAO_EMS_726313,PT,FAO_ITEM_6517,1990,21.0300,Pre- and Post-Production,FAO


## Step 5 — Clean the JRC Dataset

In [11]:
jrc = jrc_raw.copy()

# ── 5a. Drop metadata columns ────────────────────────────────────
jrc_drop = [
    'FREQ', 'FREQ_LABEL',
    'REF_AREA_LABEL',
    'INDICATOR_LABEL',
    'UNIT_MEASURE_LABEL',
    'COMP_BREAKDOWN_1_LABEL',
    'AGG_METHOD', 'AGG_METHOD_LABEL',
    'DECIMALS', 'DECIMALS_LABEL',
    'DATABASE_ID', 'DATABASE_ID_LABEL',
    'UNIT_MULT', 'UNIT_MULT_LABEL',
    'DATA_SOURCE', 'DATA_SOURCE_LABEL',
    'OBS_STATUS', 'OBS_STATUS_LABEL',
    'OBS_CONF', 'OBS_CONF_LABEL'
]
jrc = jrc.drop(columns=jrc_drop)

# ── 5b. Keep only total rows (remove mercury sub-species) ────────
# _T = Total; HG_D, HG_G, HG_P are mercury sub-species breakdowns
# Remove sub-species to avoid double-counting in any aggregation
jrc = jrc[jrc['COMP_BREAKDOWN_1'] == '_T']
print(f'JRC rows after keeping _T only: {len(jrc)}')

# ── 5c. Melt wide → long ─────────────────────────────────────────
id_vars_jrc = ['REF_AREA', 'INDICATOR', 'UNIT_MEASURE', 'COMP_BREAKDOWN_1']
year_cols   = [c for c in jrc.columns if c not in id_vars_jrc]

jrc_long = jrc.melt(
    id_vars   = id_vars_jrc,
    value_vars = year_cols,
    var_name   = 'year',
    value_name = 'value'
)

# ── 5d. Fix data types ───────────────────────────────────────────
jrc_long['year']  = jrc_long['year'].astype(int)
jrc_long['value'] = pd.to_numeric(jrc_long['value'], errors='coerce')

# ── 5e. Drop nulls ───────────────────────────────────────────────
before = len(jrc_long)
jrc_long = jrc_long.dropna(subset=['value'])
print(f'JRC rows dropped (null): {before - len(jrc_long)} ({(before-len(jrc_long))/before*100:.1f}%)')
print(f'JRC long shape: {jrc_long.shape}')

JRC rows after keeping _T only: 2194
JRC rows dropped (null): 560 (0.5%)
JRC long shape: (115722, 6)


In [12]:
# ── 5f. Add readable pollutant label ─────────────────────────────
pollutant_labels = {
    'JRC_EDGAR_BC':    'Black Carbon (BC)',
    'JRC_EDGAR_CO':    'Carbon Monoxide (CO)',
    'JRC_EDGAR_HG':    'Mercury (Hg)',
    'JRC_EDGAR_NH3':   'Ammonia (NH3)',
    'JRC_EDGAR_NMVOC': 'NM Volatile Organics (NMVOC)',
    'JRC_EDGAR_NOX':   'Nitrogen Oxides (NOx)',
    'JRC_EDGAR_OC':    'Organic Carbon (OC)',
    'JRC_EDGAR_PM10':  'Fine Particles PM10',
    'JRC_EDGAR_PM2_5': 'Fine Particles PM2.5',
    'JRC_EDGAR_SO2':   'Sulfur Dioxide (SO2)'
}
jrc_long['sector'] = jrc_long['INDICATOR'].map(pollutant_labels)

# ── 5g. Add source tag ───────────────────────────────────────────
jrc_long['source'] = 'JRC_EDGAR'

print('JRC cleaned. Sample:')
jrc_long.head(3)

JRC cleaned. Sample:


,REF_AREA,INDICATOR,UNIT_MEASURE,COMP_BREAKDOWN_1,year,value,sector,source
0,ABW,JRC_EDGAR_BC,T,_T,1970,0.0073,Black Carbon (BC),JRC_EDGAR
1,ABW,JRC_EDGAR_CO,T,_T,1970,0.2802,Carbon Monoxide (CO),JRC_EDGAR
2,ABW,JRC_EDGAR_HG,T,_T,1970,0.0000,Mercury (Hg),JRC_EDGAR


## Step 6 — Join the Datasets

### Option A — Vertical stack (recommended for dashboarding)
Both datasets share the same schema after cleaning. Stack them into one combined dataframe.

In [13]:
# ── Option A: vertical concat ────────────────────────────────────
combined = pd.concat([fao_long, jrc_long], ignore_index=True)

print('Combined shape:', combined.shape)
print('Source breakdown:')
print(combined['source'].value_counts())
print('\nSchema:')
print(combined.dtypes)
combined.head(5)

Combined shape: (210322, 8)
Source breakdown:
source
JRC_EDGAR    115722
FAO           94600
Name: count, dtype: int64

Schema:
REF_AREA             object
INDICATOR            object
UNIT_MEASURE         object
COMP_BREAKDOWN_1     object
year                  int64
value               float64
sector               object
source               object
dtype: object


,REF_AREA,INDICATOR,UNIT_MEASURE,COMP_BREAKDOWN_1,year,value,sector,source
0,ABW,FAO_EMS_726313,PT,FAO_ITEM_1707,1990,0.0600,LULUCF,FAO
1,ABW,FAO_EMS_726313,PT,FAO_ITEM_1711,1990,0.0000,IPCC Agriculture,FAO
2,ABW,FAO_EMS_726313,PT,FAO_ITEM_6517,1990,21.0300,Pre- and Post-Production,FAO
3,ABW,FAO_EMS_726313,PT,FAO_ITEM_6518,1990,21.0300,Agrifood systems,FAO
4,ABW,FAO_EMS_726313,PT,FAO_ITEM_6818,1990,2.5400,Waste,FAO


In [14]:
# ── Option B: horizontal merge (pivot + merge) ───────────────────
# Use this when you want to compare FAO % share vs JRC pollutant
# tonnes side-by-side for the same country and year

# Filter FAO to a single summary sector for the pivot
# Using 'All sectors with LULUCF' as the overall agrifood share
fao_summary = fao_long[fao_long['COMP_BREAKDOWN_1'] == 'FAO_ITEM_6825'][['REF_AREA','year','value']]
fao_summary = fao_summary.rename(columns={'value': 'agrifood_share_pct'})

# Pivot JRC to get one column per pollutant
jrc_pivot = jrc_long.pivot_table(
    index   = ['REF_AREA', 'year'],
    columns = 'INDICATOR',
    values  = 'value'
).reset_index()
jrc_pivot.columns.name = None

# Rename JRC columns to friendly names
jrc_pivot = jrc_pivot.rename(columns={
    k: v.split('(')[0].strip().lower().replace(' ', '_')
    for k, v in pollutant_labels.items()
})

# Merge on country + year (inner = common countries, years 1990-2022)
merged = pd.merge(fao_summary, jrc_pivot, on=['REF_AREA','year'], how='inner')

print('Merged (horizontal) shape:', merged.shape)
merged.head()

Merged (horizontal) shape: (6474, 13)


,REF_AREA,year,agrifood_share_pct,black_carbon,carbon_monoxide,mercury,ammonia,nm_volatile_organics,nitrogen_oxides,organic_carbon,fine_particles_pm10,fine_particles_pm2.5,sulfur_dioxide
0,ABW,1990,100.0000,0.0189,3.2789,0.0000,0.0689,0.6650,1.1649,0.0141,0.0964,0.0672,1.9572
1,AFG,1990,100.0000,0.7905,98.6805,0.0001,46.3279,39.3343,18.4802,3.4026,8.9638,6.7363,16.7921
2,AGO,1990,100.0000,9.7665,1106.5905,0.0004,54.9994,221.3355,36.8144,29.8548,151.8788,61.1599,19.0276
3,AIA,1990,100.0000,0.0008,0.2609,0.0000,0.0002,0.0538,0.0654,0.0006,0.0024,0.0020,0.0158
4,ALB,1990,100.0000,1.7358,134.3816,0.0002,38.4526,40.3569,25.8853,6.3317,17.7361,15.7382,28.7326


### Visualizing the Data Join & Year Filter

To confirm the join logic, we can verify that the years in our merged dataset are strictly within the intersection of the FAO and JRC timelines (1990 to 2022).

In [15]:
# 1. Identify the overlap
fao_years = set(fao_long['year'].unique())
jrc_years = set(jrc_long['year'].unique())
overlap = sorted(list(fao_years & jrc_years))

print(f"FAO Year Range: {min(fao_years)} - {max(fao_years)}")
print(f"JRC Year Range: {min(jrc_years)} - {max(jrc_years)}")
print(f"Overlap Range:   {min(overlap)} - {max(overlap)}")

# 2. Check the merged dataframe
print(f"\nMerged DataFrame Year Range: {merged['year'].min()} - {merged['year'].max()}")

# 3. Visualize row counts per year to see the 'Joining' effect
year_counts = merged['year'].value_counts().sort_index().reset_index()
year_counts.columns = ['year', 'record_count']

fig = px.bar(
    year_counts,
    x='year',
    y='record_count',
    title='Records per Year in Joined Dataset (1990-2022)',
    labels={'record_count': 'Number of Countries', 'year': 'Year'},
    width=800, height=400
)
fig.show()

FAO Year Range: 1990 - 2023
JRC Year Range: 1970 - 2022
Overlap Range:   1990 - 2022

Merged DataFrame Year Range: 1990 - 2022


/usr/local/lib/python3.12/dist-packages/kaleido/_sync_server.py:11: UserWarning:




This means that static image generation (e.g. `fig.write_image()`) will not work.

Please upgrade Plotly to version 6.1.1 or greater, or downgrade Kaleido to version 0.2.1.




## Step 7 — Export Clean Files

In [16]:
fao_long.to_csv('fao_clean.csv', index=False)
jrc_long.to_csv('jrc_clean.csv', index=False)
combined.to_csv('emissions_combined.csv', index=False)
merged.to_csv('emissions_merged_horizontal.csv', index=False)

print('Files saved:')
print('  fao_clean.csv                  — cleaned FAO long format')
print('  jrc_clean.csv                  — cleaned JRC long format')
print('  emissions_combined.csv         — stacked (Option A)')
print('  emissions_merged_horizontal.csv — side-by-side (Option B)')

# Download from Colab (uncomment if needed)
# from google.colab import files
# files.download('emissions_combined.csv')

Files saved:
  fao_clean.csv                  — cleaned FAO long format
  jrc_clean.csv                  — cleaned JRC long format
  emissions_combined.csv         — stacked (Option A)
  emissions_merged_horizontal.csv — side-by-side (Option B)


## Step 8 — Dashboard Charts (Plotly)

### Chart 1: Global choropleth — agrifood emissions share by country

In [17]:
# Agrifood emissions share — 'All sectors with LULUCF'
df_map = fao_long[
    (fao_long['COMP_BREAKDOWN_1'] == 'FAO_ITEM_6825') &
    (fao_long['year'].between(1990, 2022))
].copy()

fig = px.choropleth(
    df_map,
    locations      = 'REF_AREA',
    color          = 'value',
    animation_frame= 'year',
    color_continuous_scale = 'YlOrRd',
    range_color    = [0, 80],
    title          = 'Agrifood Systems Emissions Share (% of total CO₂eq) — 1990 to 2022',
    labels         = {'value': '% Share', 'REF_AREA': 'Country'},
    width=900, height=500
)
fig.update_layout(coloraxis_colorbar=dict(title='% Share'))
fig.show()

### Chart 2: Country deep-dive — sector breakdown over time

In [18]:
# Change COUNTRY_CODE to any REF_AREA code, e.g. 'IND', 'BRA', 'CHN', 'LKA'
COUNTRY_CODE = 'IND'

# Exclude aggregate rows to show only individual sectors
exclude_sectors = ['FAO_ITEM_6825', 'FAO_ITEM_6829', 'FAO_ITEM_6518', 'FAO_ITEM_6824']

df_country = fao_long[
    (fao_long['REF_AREA'] == COUNTRY_CODE) &
    (~fao_long['COMP_BREAKDOWN_1'].isin(exclude_sectors))
].copy()

country_name = fao_raw[fao_raw['REF_AREA'] == COUNTRY_CODE]['REF_AREA_LABEL'].iloc[0]

fig = px.area(
    df_country,
    x     = 'year',
    y     = 'value',
    color = 'sector',
    title = f'{country_name} — Agrifood Sector Emissions Share Over Time',
    labels= {'value': '% of total CO₂eq', 'year': 'Year', 'sector': 'Sector'},
    width=860, height=450
)
fig.update_layout(hovermode='x unified')
fig.show()

### Demonstrating Sector Separation
To follow the best practice of avoiding double-counting, we can create separate dataframes for 'Total' aggregates and 'Sub-sector' details.

In [19]:
# Define our aggregate sector codes
aggregates = ['FAO_ITEM_6825', 'FAO_ITEM_6829', 'FAO_ITEM_6518', 'FAO_ITEM_6824']

# Create a dataframe for Top-level totals (Aggregates)
df_aggregates = fao_long[fao_long['COMP_BREAKDOWN_1'].isin(aggregates)].copy()

# Create a dataframe for granular Sub-sectors (Detailed breakdown)
df_details = fao_long[~fao_long['COMP_BREAKDOWN_1'].isin(aggregates)].copy()

print(f"Total rows in dataset: {len(fao_long)}")
print(f"Rows in Aggregate view: {len(df_aggregates)}")
print(f"Rows in Detail view:    {len(df_details)}")

# Quick check to ensure no overlap in a chart
display(df_details['sector'].unique())

Total rows in dataset: 94600
Rows in Aggregate view: 27288
Rows in Detail view:    67312


array(['LULUCF', 'IPCC Agriculture', 'Pre- and Post-Production', 'Waste',
       'Other', 'Energy', 'Emissions on agricultural land', 'Farm gate',
       'Emissions from crops', 'Emissions from livestock'], dtype=object)

### Chart 3: Air pollutant trend — multi-line for one country

In [20]:
# Change COUNTRY_CODE to match chart 2 for a side-by-side story
COUNTRY_CODE = 'IND'
country_name = fao_raw[fao_raw['REF_AREA'] == COUNTRY_CODE]['REF_AREA_LABEL'].iloc[0]

df_jrc_country = jrc_long[jrc_long['REF_AREA'] == COUNTRY_CODE].copy()

# Exclude mercury (very small scale) for readability
df_jrc_country = df_jrc_country[df_jrc_country['INDICATOR'] != 'JRC_EDGAR_HG']

fig = px.line(
    df_jrc_country,
    x     = 'year',
    y     = 'value',
    color = 'sector',
    title = f'{country_name} — Air Pollutant Emissions (tonnes, 1970–2022)',
    labels= {'value': 'Tonnes', 'year': 'Year', 'sector': 'Pollutant'},
    width=860, height=450
)
fig.update_layout(hovermode='x unified')
fig.show()

### Chart 4: NH₃ vs agrifood share scatter — finding the agriculture-pollution signal

In [21]:
# Use the horizontal merged dataframe from Step 6 Option B
# Filter to a single year for clarity
YEAR = 2019

df_scatter = merged[merged['year'] == YEAR].copy()

# Add country names
country_lookup = fao_raw[['REF_AREA','REF_AREA_LABEL']].drop_duplicates()
df_scatter = df_scatter.merge(country_lookup, on='REF_AREA', how='left')

# We need the NH3 column name — check what it was renamed to in Option B
nh3_col = [c for c in df_scatter.columns if 'ammonia' in c.lower() or 'nh3' in c.lower()]
if not nh3_col:
    # Fallback: use original indicator column if rename wasn't applied
    nh3_col_name = 'JRC_EDGAR_NH3'
else:
    nh3_col_name = nh3_col[0]

fig = px.scatter(
    df_scatter,
    x         = 'agrifood_share_pct',
    y         = nh3_col_name,
    hover_name= 'REF_AREA_LABEL',
    size      = nh3_col_name,
    color     = 'agrifood_share_pct',
    color_continuous_scale = 'Viridis',
    log_y     = True,
    title     = f'Agrifood Emissions Share vs NH₃ Emissions by Country ({YEAR})',
    labels    = {
        'agrifood_share_pct': 'Agrifood Share (% CO₂eq)',
        nh3_col_name: 'NH₃ Emissions (tonnes, log scale)'
    },
    width=860, height=520
)
fig.show()
# Insight: countries with high agrifood share cluster toward higher NH3 —
# this confirms livestock/crop emissions as a primary NH3 driver.

### Chart 5: Top 10 improvers — % reduction in PM2.5 since 1990

In [22]:
import plotly.io as pio
import plotly.express as px

# 1. Force the Colab-specific renderer
pio.renderers.default = 'colab'

# Calculate % change in PM2.5 from 1990 baseline to most recent year
pm25 = jrc_long[jrc_long['INDICATOR'] == 'JRC_EDGAR_PM2_5'].copy()

baseline = pm25[pm25['year'] == 1990][['REF_AREA', 'value']].rename(columns={'value': 'val_1990'})
latest   = pm25[pm25['year'] == 2022][['REF_AREA', 'value']].rename(columns={'value': 'val_2022'})

progress = baseline.merge(latest, on='REF_AREA')
progress['pct_change'] = (progress['val_2022'] - progress['val_1990']) / progress['val_1990'] * 100

# Add country names
country_lookup = fao_raw[['REF_AREA', 'REF_AREA_LABEL']].drop_duplicates()
progress = progress.merge(country_lookup, on='REF_AREA', how='left')

# Top 10 improvers
top10 = progress.nsmallest(10, 'pct_change')

if top10.empty:
    print('No data found for the selected years.')
else:
    fig = px.bar(
        top10,
        x='pct_change',
        y='REF_AREA_LABEL',
        orientation='h',
        color='pct_change',
        color_continuous_scale='RdYlGn',
        title='Top 10 countries by PM2.5 reduction since 1990 (%)',
        labels={'pct_change': '% change', 'REF_AREA_LABEL': ''},
        width=760, height=420
    )
    fig.update_layout(yaxis={'categoryorder': 'total ascending'}, showlegend=False)

    # Attempt to show interactive first
    fig.show()

    # Print raw data as fallback so the user isn't left with nothing
    print('\n--- Raw Data Summary ---')
    display(top10[['REF_AREA_LABEL', 'pct_change']])


--- Raw Data Summary ---


,REF_AREA_LABEL,pct_change
67,NaN,-94.8936
127,Malta,-89.0793
106,St. Kitts and Nevis,-88.8943
46,Cuba,-81.1922
85,"Hong Kong SAR, China",-74.1645
49,Czechia,-70.1401
134,Mauritius,-69.2357
70,Georgia,-67.8172
50,Germany,-66.4295
8,Armenia,-64.5587


### Handling Scale Differences: Dual-Axis Comparison

To compare FAO (percentages) and JRC (tonnes) accurately, we use a dual y-axis chart. This prevents the percentage data from appearing as a flat line at the bottom of the chart.

In [23]:
# Fix corrupted Plotly installation
!pip uninstall -y plotly
!pip install plotly==5.24.1 --quiet

import plotly.graph_objects as go
from plotly.subplots import make_subplots

# Re-running the dual-axis chart code
COUNTRY = "IND"
pollutant_col = "ammonia"
df_dual = merged[merged["REF_AREA"] == COUNTRY].copy()

fig = make_subplots(specs=[[{"secondary_y": True}]])

fig.add_trace(
    go.Scatter(x=df_dual["year"], y=df_dual["agrifood_share_pct"], name="Agrifood Share (%)", mode="lines+markers"),
    secondary_y=False,
)

fig.add_trace(
    go.Scatter(x=df_dual["year"], y=df_dual[pollutant_col], name=f"{pollutant_col.title()} (Tonnes)", mode="lines+markers"),
    secondary_y=True,
)

fig.update_layout(
    title_text=f"Scale Comparison for {COUNTRY}: Agrifood Share vs {pollutant_col.title()}",
    width=900, height=500
)

fig.update_yaxes(title_text="<b>FAO</b> Share (%)", secondary_y=False)
fig.update_yaxes(title_text=f"<b>JRC</b> {pollutant_col.title()} (Tonnes)", secondary_y=True)

fig.show()

Found existing installation: plotly 5.24.1
Uninstalling plotly-5.24.1:
  Successfully uninstalled plotly-5.24.1
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 19.1/19.1 MB 74.3 MB/s eta 0:00:00


In [24]:
!pip install pycountry

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 8.0/8.0 MB 98.4 MB/s eta 0:00:00


## Notes & next steps

- **Year filter for joined analysis:** FAO runs 1990–2023, JRC runs 1970–2022. For cross-dataset comparisons, filter to **1990–2022** (the overlapping window).
- **Sector aggregation tip:** Avoid mixing aggregate FAO rows (e.g. `FAO_ITEM_6825` = all sectors) with sub-sector rows in the same chart — it will double-count emissions.
- **Scale difference:** FAO values are percentages (0–100), JRC values are absolute tonnes (can be in millions). Never plot them on the same y-axis without normalisation — use dual-axis or separate panels.
- **Streamlit deployment:** To turn these charts into a live web app, copy the chart code into a `streamlit` script and use `st.plotly_chart(fig)`. Free hosting at streamlit.io/cloud.


### Preparing for Streamlit Deployment

To move from a notebook to a web app, we can write our logic into a single Python file. Below is an example of how to structure `app.py` using the data we've cleaned.

In [25]:
# ── 3. Write the dashboard script ──────────────────────────────────────────
dashboard_code = '''
import streamlit as st
import pandas as pd
import plotly.express as px
import plotly.graph_objects as go
from plotly.subplots import make_subplots
import numpy as np

st.set_page_config(
    page_title="Agrifood Emissions Dashboard",
    page_icon="\U0001f331",
    layout="wide",
    initial_sidebar_state="expanded",
)

st.markdown(\'\'\'<style>
    [data-testid="stSidebar"] { background-color: #1a2433; }
    [data-testid="stSidebar"] p,
    [data-testid="stSidebar"] span,
    [data-testid="stSidebar"] div,
    [data-testid="stSidebar"] label,
    [data-testid="stSidebar"] h1,
    [data-testid="stSidebar"] h2,
    [data-testid="stSidebar"] h3 { color: #dce6f0 !important; }
    [data-testid="stSidebar"] .stSelectbox label,
    [data-testid="stSidebar"] .stMultiselect label,
    [data-testid="stSidebar"] .stSlider label { color: #9eb3cc !important; font-size: 0.82rem; }
    [data-testid="metric-container"] {
        background: #f7f9fc; border: 1px solid #e3e8f0;
        border-radius: 10px; padding: 12px 16px;
    }
    [data-testid="stMetricValue"] { font-size: 1.5rem !important; font-weight: 700; color: #1a2433 !important; }
    [data-testid="stMetricLabel"] { font-size: 0.78rem !important; color: #4a5568 !important; }
    .section-header {
        font-size: 1.05rem; font-weight: 700; color: #1a2433;
        border-left: 4px solid #2e86de; padding-left: 10px; margin: 18px 0 10px 0;
    }
    .info-box {
        background: #2c3e55; border-radius: 8px; padding: 10px 14px;
        font-size: 0.83rem; color: #cfe0f0 !important; margin-bottom: 8px;
    }
    div.block-container { padding-top: 1.5rem; }
    .main .block-container, .main p, .main span, .main div { color: #1a2433; }
</style>\'\'\', unsafe_allow_html=True)

POLLUTANT_LABELS = {
    "black_carbon": "Black Carbon (Gg)",
    "carbon_monoxide": "Carbon Monoxide (Gg)",
    "mercury": "Mercury (Gg)",
    "ammonia": "Ammonia (Gg)",
    "nm_volatile_organics": "NM Volatile Organics (Gg)",
    "nitrogen_oxides": "Nitrogen Oxides (Gg)",
    "organic_carbon": "Organic Carbon (Gg)",
    "fine_particles_pm10": "Fine Particles PM10 (Gg)",
    "fine_particles_pm2.5": "Fine Particles PM2.5 (Gg)",
    "sulfur_dioxide": "Sulfur Dioxide (Gg)",
}
POLLUTANT_COLS = list(POLLUTANT_LABELS.keys())

import pycountry

def get_country_names():
    names = {}
    for country in pycountry.countries:
        if hasattr(country, 'alpha_3'):
            names[country.alpha_3] = country.name
    # Manual overrides for cleaner display names
    names.update({
        "USA": "United States",
        "GBR": "United Kingdom",
        "IRN": "Iran",
        "RUS": "Russia",
        "KOR": "South Korea",
        "PRK": "North Korea",
        "TZA": "Tanzania",
        "MDA": "Moldova",
        "SYR": "Syria",
        "VEN": "Venezuela",
        "BOL": "Bolivia",
    })
    return names

COUNTRY_NAMES = get_country_names()
}

@st.cache_data
def load_data():
    df = pd.read_csv("emissions_merged_horizontal.csv")
    df["year"] = df["year"].astype(int)
    for col in POLLUTANT_COLS:
        df[col] = pd.to_numeric(df[col], errors="coerce")
    df["total_emissions"] = df[POLLUTANT_COLS].sum(axis=1)
    df["display_name"] = df["REF_AREA"].map(COUNTRY_NAMES).fillna(df["REF_AREA"])
    return df

df = load_data()
all_countries = sorted(df["REF_AREA"].unique())
year_min, year_max = int(df["year"].min()), int(df["year"].max())

with st.sidebar:
    st.markdown("## \U0001f331 Dashboard Controls")
    st.markdown("---")
    st.markdown("**Country / Region**")
    default_idx = all_countries.index("IND") if "IND" in all_countries else 0
    selected_country = st.selectbox("", all_countries, index=default_idx, label_visibility="collapsed")
    st.markdown("**Year Range**")
    year_range = st.slider("", year_min, year_max, (1995, 2022), label_visibility="collapsed")
    st.markdown("**Primary Pollutant**")
    pollutant_display = {v: k for k, v in POLLUTANT_LABELS.items()}
    selected_pollutant_label = st.selectbox("", list(POLLUTANT_LABELS.values()), label_visibility="collapsed")
    selected_pollutant = pollutant_display[selected_pollutant_label]
    st.markdown("**Comparison Countries**")
    compare_defaults = [c for c in ["USA","CHN","BRA","DEU"] if c in all_countries and c != selected_country][:3]
    compare_countries = st.multiselect("", all_countries, default=[selected_country]+compare_defaults, label_visibility="collapsed")
    st.markdown("---")
    st.markdown(\'<div class="info-box">Data: FAO Agrifood + JRC Air Pollutants<br>Coverage: 200+ countries · 1990\xe2\x80\x932022</div>\', unsafe_allow_html=True)

mask = (df["REF_AREA"] == selected_country) & df["year"].between(*year_range)
cdf = df[mask].copy()
country_label = COUNTRY_NAMES.get(selected_country, selected_country)

st.markdown(f"## \U0001f30d Agrifood Emissions \u2014 **{country_label}** ({selected_country})")
st.markdown(f"*Exploring air pollutant trends from agrifood systems · {year_range[0]}\u2013{year_range[1]}*")
st.markdown("---")

if not cdf.empty:
    latest = cdf[cdf["year"] == cdf["year"].max()].iloc[0]
    earliest = cdf[cdf["year"] == cdf["year"].min()].iloc[0]
    def safe_delta(a, b):
        return f"{((a-b)/b*100):.1f}%" if b and b!=0 else "N/A"
    c1,c2,c3,c4,c5 = st.columns(5)
    c1.metric("\U0001f4c5 Latest Year", str(int(latest["year"])))
    c2.metric("\U0001f3ed Total Emissions", f"{latest[\'total_emissions\']:,.0f} Gg", delta=safe_delta(latest[\'total_emissions\'],earliest[\'total_emissions\']))
    c3.metric(f"\U0001f4cc {selected_pollutant_label.split(chr(32))[0]}", f"{latest[selected_pollutant]:,.2f}", delta=safe_delta(latest[selected_pollutant],earliest[selected_pollutant]))
    c4.metric("\U0001f321 Agrifood Share", f"{latest[\'agrifood_share_pct\']:.1f}%")
    c5.metric("\U0001f4ca Data Points", len(cdf))

st.markdown("---")
tab1,tab2,tab3,tab4 = st.tabs(["\U0001f4c8 Trend Analysis","\U0001f52c Pollutant Breakdown","\U0001f310 Global Comparison","\U0001f4cb Data Table"])

with tab1:
    if cdf.empty:
        st.warning("No data available.")
    else:
        st.markdown(\'<div class="section-header">Total Emissions vs Selected Pollutant Over Time</div>\', unsafe_allow_html=True)
        fig_dual = make_subplots(specs=[[{"secondary_y": True}]])
        fig_dual.add_trace(go.Scatter(x=cdf["year"],y=cdf["total_emissions"],name="Total Emissions (Gg)",mode="lines+markers",line=dict(color="#2e86de",width=2.5),marker=dict(size=5)),secondary_y=False)
        fig_dual.add_trace(go.Scatter(x=cdf["year"],y=cdf[selected_pollutant],name=selected_pollutant_label,mode="lines+markers",line=dict(color="#e84118",width=2.5,dash="dot"),marker=dict(size=5)),secondary_y=True)
        fig_dual.update_layout(hovermode="x unified",legend=dict(orientation="h",yanchor="bottom",y=1.02,xanchor="right",x=1),margin=dict(l=0,r=0,t=40,b=0),height=360,plot_bgcolor="#ffffff",paper_bgcolor="#ffffff")
        fig_dual.update_yaxes(title_text="Total Emissions (Gg)",secondary_y=False,gridcolor="#e8edf3")
        fig_dual.update_yaxes(title_text=selected_pollutant_label,secondary_y=True)
        fig_dual.update_xaxes(gridcolor="#e8edf3")
        st.plotly_chart(fig_dual,use_container_width=True)

        st.markdown(\'<div class="section-header">Stacked Pollutant Composition Over Time</div>\', unsafe_allow_html=True)
        area_df = cdf[["year"]+POLLUTANT_COLS].melt("year",var_name="pollutant",value_name="value")
        area_df["pollutant_label"] = area_df["pollutant"].map(POLLUTANT_LABELS)
        area_df = area_df.dropna(subset=["value"])
        fig_area = px.area(area_df,x="year",y="value",color="pollutant_label",color_discrete_sequence=px.colors.qualitative.Safe,labels={"value":"Emissions (Gg)","year":"Year","pollutant_label":"Pollutant"})
        fig_area.update_layout(hovermode="x unified",height=320,legend=dict(orientation="h",yanchor="bottom",y=1.01,xanchor="right",x=1),margin=dict(l=0,r=0,t=40,b=0),plot_bgcolor="#ffffff",paper_bgcolor="#ffffff")
        fig_area.update_xaxes(gridcolor="#e8edf3")
        fig_area.update_yaxes(gridcolor="#e8edf3")
        st.plotly_chart(fig_area,use_container_width=True)

with tab2:
    if cdf.empty:
        st.warning("No data.")
    else:
        col_a,col_b = st.columns(2)
        with col_a:
            st.markdown(\'<div class="section-header">Pollutant Share (Latest Year)</div>\', unsafe_allow_html=True)
            lr = cdf[cdf["year"]==cdf["year"].max()].iloc[0]
            pv = {POLLUTANT_LABELS[p]:lr[p] for p in POLLUTANT_COLS if pd.notna(lr[p]) and lr[p]>0}
            fig_pie = px.pie(names=list(pv.keys()),values=list(pv.values()),color_discrete_sequence=px.colors.qualitative.Safe,hole=0.38)
            fig_pie.update_traces(textposition="inside",textinfo="percent+label")
            fig_pie.update_layout(showlegend=False,height=340,margin=dict(l=0,r=0,t=20,b=0))
            st.plotly_chart(fig_pie,use_container_width=True)
        with col_b:
            st.markdown(\'<div class="section-header">Average Annual Emissions by Pollutant</div>\', unsafe_allow_html=True)
            av = cdf[POLLUTANT_COLS].mean().reset_index()
            av.columns=["pollutant","avg"]
            av["label"]=av["pollutant"].map(POLLUTANT_LABELS)
            av=av.sort_values("avg",ascending=True)
            fig_bar=px.bar(av,x="avg",y="label",orientation="h",color="avg",color_continuous_scale="Blues",labels={"avg":"Avg Emissions (Gg)","label":""})
            fig_bar.update_layout(coloraxis_showscale=False,height=340,margin=dict(l=0,r=0,t=20,b=0),plot_bgcolor="#ffffff",paper_bgcolor="#ffffff")
            fig_bar.update_xaxes(gridcolor="#e8edf3")
            st.plotly_chart(fig_bar,use_container_width=True)

        st.markdown(\'<div class="section-header">Correlation Matrix</div>\', unsafe_allow_html=True)
        cd = cdf[POLLUTANT_COLS].dropna(how="all").rename(columns=POLLUTANT_LABELS)
        fig_heat=px.imshow(cd.corr(),text_auto=".2f",color_continuous_scale="RdBu_r",zmin=-1,zmax=1)
        fig_heat.update_layout(height=380,margin=dict(l=0,r=0,t=20,b=0))
        st.plotly_chart(fig_heat,use_container_width=True)

        st.markdown(f\'<div class="section-header">{selected_pollutant_label} vs Total Emissions</div>\', unsafe_allow_html=True)
        fig_sc=px.scatter(cdf,x="total_emissions",y=selected_pollutant,color="year",size=selected_pollutant,size_max=20,hover_data=["year"],color_continuous_scale="Viridis",trendline="ols",labels={"total_emissions":"Total Emissions (Gg)",selected_pollutant:selected_pollutant_label,"year":"Year"})
        fig_sc.update_layout(height=320,margin=dict(l=0,r=0,t=20,b=0),plot_bgcolor="#ffffff",paper_bgcolor="#ffffff")
        fig_sc.update_xaxes(gridcolor="#e8edf3")
        fig_sc.update_yaxes(gridcolor="#e8edf3")
        st.plotly_chart(fig_sc,use_container_width=True)

with tab3:
    if not compare_countries:
        st.info("Select countries in the sidebar.")
    else:
        cmp=df[df["REF_AREA"].isin(compare_countries)&df["year"].between(*year_range)].copy()
        cmp["label"]=cmp["REF_AREA"].map(COUNTRY_NAMES).fillna(cmp["REF_AREA"])
        st.markdown(\'<div class="section-header">Selected Pollutant Trend</div>\', unsafe_allow_html=True)
        fig_cmp=px.line(cmp,x="year",y=selected_pollutant,color="label",markers=True,color_discrete_sequence=px.colors.qualitative.Bold,labels={"label":"Country",selected_pollutant:selected_pollutant_label,"year":"Year"})
        fig_cmp.update_layout(hovermode="x unified",height=340,legend=dict(orientation="h",yanchor="bottom",y=1.02,xanchor="right",x=1),margin=dict(l=0,r=0,t=40,b=0),plot_bgcolor="#ffffff",paper_bgcolor="#ffffff")
        fig_cmp.update_xaxes(gridcolor="#e8edf3")
        fig_cmp.update_yaxes(gridcolor="#e8edf3")
        st.plotly_chart(fig_cmp,use_container_width=True)

        lyr=cmp["year"].max()
        bdf=cmp[cmp["year"]==lyr][["label","total_emissions"]+POLLUTANT_COLS]
        bm=bdf.melt("label",POLLUTANT_COLS,var_name="pollutant",value_name="value")
        bm["pollutant_label"]=bm["pollutant"].map(POLLUTANT_LABELS)
        st.markdown(\'<div class="section-header">Stacked Emissions — Latest Year</div>\', unsafe_allow_html=True)
        fig_gb=px.bar(bm,x="label",y="value",color="pollutant_label",barmode="stack",color_discrete_sequence=px.colors.qualitative.Safe,labels={"value":"Emissions (Gg)","label":"Country","pollutant_label":"Pollutant"})
        fig_gb.update_layout(height=340,legend=dict(orientation="h",yanchor="bottom",y=1.01,xanchor="right",x=1),margin=dict(l=0,r=0,t=50,b=0),plot_bgcolor="#ffffff",paper_bgcolor="#ffffff")
        fig_gb.update_xaxes(gridcolor="#e8edf3")
        fig_gb.update_yaxes(gridcolor="#e8edf3")
        st.plotly_chart(fig_gb,use_container_width=True)

        st.markdown(\'<div class="section-header">Pollutant Profile Radar (Normalised)</div>\', unsafe_allow_html=True)
        rdf=bdf.set_index("label")[POLLUTANT_COLS]
        rn=rdf.div(rdf.max()).fillna(0)
        cats=[POLLUTANT_LABELS[p] for p in POLLUTANT_COLS]
        fig_rad=go.Figure()
        cr=px.colors.qualitative.Bold
        for i,c in enumerate(rn.index):
            v=rn.loc[c].tolist()
            v+=v[:1]
            fig_rad.add_trace(go.Scatterpolar(r=v,theta=cats+cats[:1],fill="toself",name=c,opacity=0.55,line=dict(color=cr[i%len(cr)],width=2)))
        fig_rad.update_layout(polar=dict(radialaxis=dict(visible=True,range=[0,1])),showlegend=True,height=400,margin=dict(l=20,r=20,t=20,b=20))
        st.plotly_chart(fig_rad,use_container_width=True)

with tab4:
    st.markdown(\'<div class="section-header">Filtered Dataset</div>\', unsafe_allow_html=True)
    sc=["REF_AREA","year","agrifood_share_pct","total_emissions"]+POLLUTANT_COLS
    sd=cdf[sc].rename(columns={"REF_AREA":"Country","year":"Year","agrifood_share_pct":"Agrifood Share (%)","total_emissions":"Total Emissions (Gg)",**POLLUTANT_LABELS})
    st.dataframe(sd.style.format({"Agrifood Share (%)":"{:.1f}","Total Emissions (Gg)":"{:,.2f}",**{v:"{:,.4f}" for v in POLLUTANT_LABELS.values()}}).background_gradient(subset=["Total Emissions (Gg)"],cmap="Blues"),use_container_width=True,height=420)
    st.download_button("\u2b07\ufe0f Download CSV",data=sd.to_csv(index=False),file_name=f"emissions_{selected_country}_{year_range[0]}_{year_range[1]}.csv",mime="text/csv")
    st.markdown(\'<div class="section-header">Summary Statistics</div>\', unsafe_allow_html=True)
    st.dataframe(sd.describe().T.style.format("{:.3f}"),use_container_width=True)
'''

with open('app.py', 'w') as f:
    f.write(dashboard_code)

print('✅ app.py written.')

✅ app.py written.


In [26]:
import textwrap

# Create requirements.txt for Streamlit deployment
requirements_content = """
pandas
plotly
kaleido
statsmodels
matplotlib
pycountry
streamlit
"""

with open('requirements.txt', 'w') as f:
    f.write(textwrap.dedent(requirements_content))

print('requirements.txt has been created.')

requirements.txt has been created.


### Deployment Steps to Streamlit Cloud

Now that you have `app.py`, `requirements.txt`, and your cleaned data files (`emissions_merged_horizontal.csv`), follow these steps to deploy your dashboard to Streamlit Cloud:

1.  **Create a GitHub Repository:**
    *   Go to [GitHub](https://github.com/) and create a new public repository.

2.  **Upload Your Files to GitHub:**
    *   Upload `app.py`, `requirements.txt`, and all generated `.csv` files (especially `emissions_merged_horizontal.csv` if your `app.py` uses it) to the root directory of your new GitHub repository.
    *   *Note: For larger datasets, consider storing them in a cloud storage solution (like Google Cloud Storage) and reading them directly from your Streamlit app, rather than committing them directly to GitHub.* For this example, committing the CSV is fine.

3.  **Deploy on Streamlit Cloud:**
    *   Go to [Streamlit Cloud](https://share.streamlit.io/).
    *   Sign in with your GitHub account.
    *   Click on the **'New app'** button.
    *   Select your GitHub repository, the main branch (e.g., `main`), and specify `app.py` as the main file path.
    *   Click **'Deploy!'**

Streamlit Cloud will then read your `requirements.txt` to install dependencies, and run your `app.py` to launch your dashboard. It might take a few minutes for the first deployment to complete.